# OuviSearch — Entrega 1: Análise Comparativa de Representações

**Desafio:** Ouvidoria Inteligente - Triagem Semântica de Manifestações Cidadãs
**Disciplina:** Tendências em Ciência da Computação
**Autor:** Pedro Nícollas Pereira Leon Lopes

Este notebook compara três formas de representar texto numericamente:
- **Bag-of-Words (BoW)**
- **TF-IDF**
- **Embeddings densos** 

Calculando a similaridade de cosseno entre pares de manifestações da Ouvidoria e discutindo as diferenças e limitações de cada abordagem.

## 1. Carregamento do corpus

In [1]:
# Conjunto de importações
import json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

with open("../data/manifestacoes.json", encoding="utf-8") as f:
    data = json.load(f)

manifestacoes = data["manifestacoes"]
ids = [m["id"] for m in manifestacoes]
textos = [m["texto"] for m in manifestacoes]
categorias = [m["categoria_oficial"] for m in manifestacoes]
idx = {doc_id: i for i, doc_id in enumerate(ids)}

print(f"{len(manifestacoes)} manifestações carregadas.")
pd.DataFrame(manifestacoes)[["id", "categoria_oficial", "data"]].head()

40 manifestações carregadas.


,id,categoria_oficial,data
0,M001,infraestrutura,2024-01-05
1,M002,saúde,2024-01-06
2,M003,infraestrutura,2024-01-07
3,M004,segurança,2024-01-08
4,M005,infraestrutura,2024-01-09


## 2. Três representações vetoriais

* **BoW** — conta a frequência bruta de cada palavra (esparsa, sem noção de importância relativa).
* **TF-IDF** — pondera termos raros no corpus com peso maior (esparsa, mas sensível à especificidade).
* **Embeddings** — representação densa aprendida por um modelo pré-treinado
  (`paraphrase-multilingual-MiniLM-L12-v2`), que captura semântica além do vocabulário exato.

Se a biblioteca `sentence-transformers` não estiver disponível no ambiente (ex.: sem acesso para
baixar o modelo), a função cai automaticamente em um **modo de simulação documentado**: uma
representação densa via TF-IDF + SVD (LSA) sobre o próprio corpus, para que o notebook continue
executável de ponta a ponta. Essa simulação é um proxy didático — como só aprende com os 40
documentos deste corpus, ela **não** tem o mesmo poder de generalização semântica de um modelo
pré-treinado em milhões de frases, como fica evidente na discussão da seção 4.

In [2]:
def carregar_modelo_embeddings(nome="paraphrase-multilingual-MiniLM-L12-v2"):
    try:
        from sentence_transformers import SentenceTransformer
        return SentenceTransformer(nome), True
    except Exception as e:
        print(f"[aviso] sentence-transformers indisponível ({e}); usando fallback TF-IDF+SVD.")
        return None, False


def gerar_embeddings(textos, nome_modelo="paraphrase-multilingual-MiniLM-L12-v2"):
    modelo, ok = carregar_modelo_embeddings(nome_modelo)
    if ok:
        vetores = modelo.encode(textos, normalize_embeddings=True)
        return np.asarray(vetores), True

    # Fallback: TF-IDF + SVD (LSA) como proxy denso, normalizado por linha (L2)
    vectorizer = TfidfVectorizer(lowercase=True, strip_accents="unicode")
    tfidf_matrix = vectorizer.fit_transform(textos)
    n_components = min(30, tfidf_matrix.shape[1] - 1, tfidf_matrix.shape[0] - 1)
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    denso = svd.fit_transform(tfidf_matrix)
    normas = np.linalg.norm(denso, axis=1, keepdims=True)
    normas[normas == 0] = 1
    return denso / normas, False


bow_vectorizer = CountVectorizer(lowercase=True, strip_accents="unicode")
bow_matrix = bow_vectorizer.fit_transform(textos).toarray()

tfidf_vectorizer = TfidfVectorizer(lowercase=True, strip_accents="unicode")
tfidf_matrix = tfidf_vectorizer.fit_transform(textos).toarray()

emb_matrix, usou_embeddings_reais = gerar_embeddings(textos)

print(f"BoW:        {bow_matrix.shape}")
print(f"TF-IDF:     {tfidf_matrix.shape}")
print(f"Embeddings: {emb_matrix.shape} (modelo real: {usou_embeddings_reais})")

/tmp/ouvisearch_test_venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  34%|███▎      | 67/199 [00:00<00:00, 645.04it/s]

Loading weights:  66%|██████▋   | 132/199 [00:00<00:00, 404.76it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 597.31it/s]

BoW:        (40, 539)
TF-IDF:     (40, 539)
Embeddings: (40, 384) (modelo real: True)


## 3. Similaridade de cosseno nos pares exigidos pelo enunciado

In [3]:
def similaridade(matriz, id_a, id_b):
    va = matriz[idx[id_a]].reshape(1, -1)
    vb = matriz[idx[id_b]].reshape(1, -1)
    return float(cosine_similarity(va, vb)[0][0])


pares = [
    ("M003", "M017", "buraco na Av. Epitácio Pessoa (Manaíra) x asfalto esburacado em Tambaú (duplicata semântica)"),
    ("M008", "M022", "posto de saúde sem médico x falta atendimento no PSF (duplicata semântica)"),
    ("M008", "M031", "posto de saúde sem médico x lâmpada queimada na praça (NÃO relacionado — controle)"),
]

linhas = []
for a, b, desc in pares:
    linhas.append({
        "Par": f"{a} × {b}",
        "Descrição": desc,
        "BoW": round(similaridade(bow_matrix, a, b), 4),
        "TF-IDF": round(similaridade(tfidf_matrix, a, b), 4),
        "Embeddings": round(similaridade(emb_matrix, a, b), 4),
    })

df_comparacao = pd.DataFrame(linhas)
df_comparacao

,Par,Descrição,BoW,TF-IDF,Embeddings
0,M003 × M017,buraco na Av. Epitácio Pessoa (Manaíra) x asfa...,0.4454,0.1558,0.5926
1,M008 × M022,posto de saúde sem médico x falta atendimento ...,0.4170,0.3087,0.7547
2,M008 × M031,posto de saúde sem médico x lâmpada queimada n...,0.3485,0.2072,0.2164


## 4. Discussão dos resultados

Resultados obtidos com o modelo real `paraphrase-multilingual-MiniLM-L12-v2` (384 dimensões):

| Par | BoW | TF-IDF | Embeddings |
|---|:---:|:---:|:---:|
| M003 × M017 (duplicata) | 0.4454 | 0.1558 | **0.5926** |
| M008 × M022 (duplicata) | 0.4170 | 0.3087 | **0.7547** |
| M008 × M031 (controle, não relacionado) | 0.3485 | 0.2072 | **0.2164** |

Achados Principais:

- **Embeddings separam duplicatadas de não-duplicatas. As representaçãos esparsas não:**

  Com embeddings, os dois pares duplicados (0.59 e 0.75) ficam muito acima do par de controle (0.22). <br>
  No BoW, a duplicata M008 × M022 (0.42) mal se distingue do par de controle (0.35). <br>
  As três manifestações compartilham muitas palavras funcionais ("na", "em", "está", "de"), então o BoW cru infla a similaridade de praticamente qualquer par de textos em português e perde poder discriminativo. <br>
  O TF-IDF corrige esse ruído (ele reduz o peso de palavras comuns), mas falha de outra forma.

<hr>

- **M003 × M017 é o caso principal:**

  Os casos **"Buraco enorme na Av. Epitácio Pessoa"** e **"asfalto todo esburacado da avenida principal"** descrevem o mesmo problema sem compartilhar nenhuma palavra de conteúdo. <br>
  O TF-IDF (0.16) dá a essa duplicata real um score *menor* que ao par de controle sem relação (0.21). Ele **inverte** o ranking esperado. <br> 
  O BoW só aparenta ir bem (0.45) pelo acúmulo de palavras funcionais, não por realmente "entender" o problema. <br>
  Já o modelo de embeddings reconhece a relação entre "buraco" e "esburacado" (0.59, quase o triplo do controle), porque foi pré-treinado em uma grande quantidade de texto e aprendeu essa proximidade semântica. Conhecimento esse que, não existe no nosso corpus de 40 documentos.

<hr>

- **Comparação com o modo de simulação:** 

  Executando este mesmo notebook sem `sentence-transformers` (fallback TF-IDF + SVD, que só aprende com o próprio corpus), os scores de Embeddings para os três pares foram 0.16, 0.56 e 0.37, ou seja, a duplicata M003 × M017 ficava novamente abaixo do controle. <br>
  Isso confirma que o ganho vem do **conhecimento pré-treinado** do modelo, e não apenas de comprimir os dados em menos dimensões.

<hr>

**Limitações de cada abordagem:**

* **BoW** -> Barato e simples, mas ignora a importância relativa das palavras e é dominado por palavras funcionais; Não distingue duplicata de texto qualquer.

* **TF-IDF** -> Pondera termos raros, mas continua exigindo sobreposição *literal* de palavras: Sinônimos e paráfrases ("buraco" × "esburacado") valem zero, podendo inverter o ranking.

* **Embeddings** -> Capturam semântica e paráfrase, mas têm custo maior (modelo de ~470MB e dependência de download) e seus scores não têm escala absoluta: Ainda que M003 × M017 seja uma duplicata real, 0.59 é um valor "médio".